# ASR performance on Italian datasets

In this notebook, the ASR performance of the models used in the ASR experimental setup with Romagnol (in asr_rgn_pipeline.ipynb) is tested on an Italian dataset. <br>

0. Imports & Functions & Global Variables
1. Prepare Italian dataset
2. ASR with baseline models
3. ASR with Romagnol fine-tuned models

Generic multilingual models (*Whisper Large*, *Whisper Medium*, and *Whisper Small*) and Italian-adapted models (*Litus-ai*, *BuzzASR*, and *Sandiago21*) are evaluated on the test set of the Mozilla Common Voice Italian dataset out of the box and after Romagnol fine-tuning to see if a better knowledge of this low-resource regional language determines a change in the ability of the models to transcribe the high-resource Italian. <br>

Both the models fine-tuned on the *shortseg* and *longseg* datasets are deployed in this pipeline, to assess whether training on short or longer segments has an impact in the performance of the models when transcribing Italian speech. <br>

The test set contains 15,183 audio clips. <br>

Evaluation metrics reported are Word Error Rate (WER) and Character Error Rate (CER).

References:

https://github.com/Mozilla-Data-Collective/datacollective-python <br>
https://huggingface.co/openai/whisper-large-v3 <br>
https://huggingface.co/openai/whisper-medium <br>
https://huggingface.co/openai/whisper-small <br>
https://huggingface.co/BuzzASR/italian?library=transformers <br>
https://huggingface.co/Sandiago21/whisper-large-v2-italian <br>
https://huggingface.co/litus-ai/whisper-small-ita <br>
https://openai.com/index/whisper/ <br>
https://github.com/SYSTRAN/faster-whisper <br>

## 0. Imports & Functions & Global Variables

In [ ]:
import os
from pathlib import Path
import sys
import wave
import jiwer
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from faster_whisper import WhisperModel
from tqdm import tqdm

In [ ]:
def transcribe_and_evaluate(
    model_or_path, df, audio_dir, language="it", beam_size=5
):
    """Transcribes audio from df col 1, evaluates against ground-truth in col 3."""
    audio_dir = Path(audio_dir)
    model = (
        WhisperModel(str(model_or_path), device="cuda", compute_type="float16")
        if isinstance(model_or_path, (str, Path))
        else model_or_path
    )

    results, refs, hyps = [], [], []

    for _, row in tqdm(
        df.iterrows(), total=len(df), desc="Transcribing & Evaluating"
    ):
        clip_name, ref = str(row.iloc[1]).strip(), str(row.iloc[3]).strip()
        stem = Path(clip_name).stem

        # Locate audio file (.wav, .mp3, or exact match)
        candidates = [
            audio_dir / clip_name,
            audio_dir / f"{stem}.wav",
            audio_dir / f"{stem}.mp3",
        ]
        audio_path = next((p for p in candidates if p.is_file()), None)
        if not audio_path:
            continue

        # Transcribe
        segments, _ = model.transcribe(
            str(audio_path), language=language, beam_size=beam_size
        )
        hyp = " ".join(s.text.strip() for s in segments).strip()

        # Metrics (lowercased)
        t_ref, t_hyp = ref.lower(), hyp.lower()
        refs.append(t_ref)
        hyps.append(t_hyp)

        results.append(
            {
                "audio_id": stem,
                "ground_truth": ref,
                "hypothesis": hyp,
                "wer": round(jiwer.wer(t_ref, t_hyp), 4) if t_ref else 0.0,
                "cer": round(jiwer.cer(t_ref, t_hyp), 4) if t_ref else 0.0,
            }
        )

    print(
        f"\nGlobal WER: {jiwer.wer(refs, hyps)*100:.2f}% | Global CER: {jiwer.cer(refs, hyps)*100:.2f}%"
    )
    return pd.DataFrame(results)

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    # Name GPU visible

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(
        f"CUDA environment ready: Using GPU 0 ({device_name}, {vram_gb:.1f} GB VRAM)."
    )
else:
    print("Warning: CUDA is still not available to PyTorch.")

CUDA environment ready: Using GPU 0 (NVIDIA RTX 5000 Ada Generation, 31.5 GB VRAM).


## 1. Prepare Italian dataset

In [3]:
test_mozilla_ita = pd.read_csv('cv-corpus-27.0-2026-09-11/it/test.tsv', sep='\t')
print(test_mozilla_ita.columns)
print(test_mozilla_ita[["path", "sentence"]])

Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')
                               path  \
0      common_voice_it_26970003.mp3   
1      common_voice_it_26970001.mp3   
2      common_voice_it_26969995.mp3   
3      common_voice_it_26970006.mp3   
4      common_voice_it_26969998.mp3   
...                             ...   
15178  common_voice_it_26970935.mp3   
15179  common_voice_it_20045040.mp3   
15180  common_voice_it_43609268.mp3   
15181  common_voice_it_40719458.mp3   
15182  common_voice_it_23606167.mp3   

                                                sentence  
0      Da questo luogo i predestinati potevano entrar...  
1      È estensivamente documentato, e la documentazi...  
2      La bambina lo fa e iniziano ad accadere molti ...  
3      Durante tale intervento furono anche aggiunte ...  
4      Il Dual Analog Controller è comp

## 2. ASR with baseline models

On MCV Italian test set, with 6 different pre-trained ASR models.

In [ ]:
# Baseline Whisper Large
local_model_path = "baseline_models/openai_whisper-large_weigths"

results_df = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(results_df.head())

In [ ]:
# Baseline Whisper Medium
local_model_path = "baseline_models/openai_whisper-medium_weigths"

base_results_medium = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(base_results_medium.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [2:16:46<00:00,  1.85it/s]  



Global WER: 13.76% | Global CER: 3.73%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0000,0.0000
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.0833,0.0115
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziano ad accadere molti ...,0.0000,0.0000
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche aggiunte ...,0.0000,0.0000
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,il dual analog controller è compatibile anche ...,0.0714,0.0109


In [ ]:
# Baseline Whisper Small
local_model_path = "baseline_models/openai_whisper-small_weigths"

base_results_small = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(base_results_small.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:13:13<00:00,  3.46it/s]



Global WER: 19.41% | Global CER: 5.38%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivamente documentato e la documentazio...,0.0833,0.0115
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziano ad accadere molti ...,0.0000,0.0000
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche aggiunte ...,0.0000,0.0000
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il Dual Analog Controller è compatibile anche ...,0.0000,0.0000


In [ ]:
# Baseline BuzzASR
local_model_path = "baseline_models/buzz_italian_ct2"

base_results_buzz = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(base_results_buzz.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [2:04:07<00:00,  2.04it/s]  



Global WER: 99.01% | Global CER: 59.66%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,From this place the predestined could enter an...,1.0000,0.6966
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",It is extensively documented and the documenta...,1.0833,0.5057
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,The child does it and many strange homicides o...,1.0000,0.6071
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,During this intervention there were also added...,1.0000,0.4074
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,The Dual Analog Controller is also compatible ...,0.7857,0.4674


In [ ]:
# Baseline Litus-ai
local_model_path = "baseline_models/litus_small_ita_ct2"

base_results_litus = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(base_results_litus.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [2:41:43<00:00,  1.56it/s]  



Global WER: 26.10% | Global CER: 9.41%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.0833,0.0115
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziano ad accadere molti ...,0.0000,0.0000
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,"Durante tale intervento, furono anche aggiunte...",0.0714,0.0093
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il dual analogue controller è compatibile anch...,0.0714,0.0217


In [ ]:
# Baseline Sandiago21
local_model_path = "baseline_models/sandiago_large_v2_it_ct2"

base_results_sandiago = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(base_results_sandiago.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [3:20:27<00:00,  1.26it/s]  



Global WER: 13.08% | Global CER: 3.54%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0000,0.0000
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.0833,0.0115
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziano ad accadere molti ...,0.0000,0.0000
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche giunte ca...,0.0714,0.0185
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il dual analog controller è compatibile anche ...,0.0000,0.0000


## 3. ASR with Romagnol fine-tuned models
Both those trained on *shortseg* and *longseg* Romagnol dataset.

In [ ]:
# Fine-tuned Whisper Large (shortseg)
local_model_path = "final_models/multi_whisp_l_shortseg_ct2_1"

ft_results_large_short = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_large_short.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [3:05:49<00:00,  1.36it/s]  



Global WER: 32.46% | Global CER: 7.91%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivamente documentato e la documentazio...,0.1667,0.0230
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina lo fa e iniziano ad accadere molti ...,0.0714,0.0119
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,durante tale intervento furono anche aggiunte ...,0.0714,0.0093
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,il dual analog controller è compatibile anche ...,0.0714,0.0109


In [ ]:
# Fine-tuned Whisper Large (longseg)
local_model_path = "final_models/multi_whisp_l_longseg_ct2_1"

ft_results_large_long = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_large_long.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:23:24<00:00,  3.03it/s]



Global WER: 18.76% | Global CER: 4.49%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivamente documentato e la documentazio...,0.1667,0.0230
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina lo fa e iniziano ad accadere molti ...,0.0714,0.0119
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche aggiunte ...,0.0000,0.0000
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il Dual Analog Controller è compatibile anche ...,0.0000,0.0000


In [ ]:
# Fine-tuned Whisper Medium (shortseg)
local_model_path = "final_models/multi_whisp_m_shortseg_ct2_1"

ft_results_medium_short = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_medium_short.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:07:29<00:00,  3.75it/s]



Global WER: 38.70% | Global CER: 10.19%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivamente documentato e la documentazio...,0.2500,0.0575
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziana ad accadere molti ...,0.0714,0.0119
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,durantetà l'intervento furon anche aggiunte ca...,0.5000,0.2778
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il Dual Analog Controller è compatibile anche ...,0.0000,0.0000


In [ ]:
# Fine-tuned Whisper Medium (longseg)
local_model_path = "final_models/multi_whisp_m_longseg_ct2_1"

ft_results_medium_long = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_medium_long.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:04:46<00:00,  3.91it/s]



Global WER: 22.55% | Global CER: 6.26%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0000,0.0000
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.1667,0.0460
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e inizia ad accadere molti st...,0.1429,0.0357
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tal intervento furono anche aggiunte c...,0.2857,0.2315
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il Dual Analog Controller è compatibile anche ...,0.0000,0.0000


In [ ]:
# Fine-tuned Whisper Small (shortseg)
local_model_path = "final_models/multi_whisp_s_shortseg_ct2_1"

ft_results_small_short = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_small_short.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [55:49<00:00,  4.53it/s] 



Global WER: 93.31% | Global CER: 29.19%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati tótéva no entrè...,0.9167,0.2360
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è e stensivamint e documentato e la documentaz...,0.9167,0.2069
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina l'ò fà e inizion a d'ac cadere molt...,0.5714,0.1190
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,durante e' tala int e' vento fura un ancagiunt...,1.0000,0.2685
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,il duval analog controler e' cumpatibili anchi...,0.5714,0.1087


In [ ]:
# Fine-tuned Whisper Small (longseg)
local_model_path = "final_models/multi_whisp_s_longseg_ct2_1"

ft_results_small_long = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)


display(ft_results_small_long.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [56:59<00:00,  4.44it/s]  



Global WER: 89.51% | Global CER: 28.51%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luògo i predestinati pò téva no entr...,1.0000,0.2135
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",e e stensivamint e documentato e la documentaz...,1.1667,0.2069
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina lo fa e' iniziava a d'acadere molti...,0.4286,0.0952
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,durante tala intervalenza fur un'ancaggiunte c...,0.9286,0.2685
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,i li du al analòch còntròllar e' còmpatibili a...,1.0000,0.2391


In [ ]:
# Fine-tuned BuzzASR (shortseg)
# local_model_path = "final_models/buzz_shortseg_ct2_1"

# ft_results_buzz_short = transcribe_and_evaluate(
#     model_or_path=local_model_path,
#     df=test_mozilla_ita,
#     audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
#     language="it",  
#     beam_size=5,
# )

# display(ft_results_buzz_short.head())

In [ ]:
# Fine-tuned BuzzASR (longseg)
# local_model_path = "final_models/buzz_longseg_ct2_1"

# ft_results_buzz_long = transcribe_and_evaluate(
#     model_or_path=local_model_path,
#     df=test_mozilla_ita,
#     audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
#     language="it",  
#     beam_size=5,
# )

# 
# display(ft_results_buzz_long.head())

In [ ]:
# Fine-tuned Litus-ai (shortseg)
local_model_path = "final_models/litus_shortseg_ct2_1"

ft_results_litus_short = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_litus_short.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [51:01<00:00,  4.96it/s] 



Global WER: 87.41% | Global CER: 28.18%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luògo i predestinati tótt éva no ent...,1.0000,0.2472
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivaménte documentato e la documentazio...,0.7500,0.1724
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina l'ò fa e inizionà da cadêri e' molt...,0.4286,0.1429
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tala e' intervënto fura un an ch'ar gi...,1.2857,0.2870
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,i du al analògh còuntròlèr e còmpatìbili anche...,0.8571,0.1957


In [ ]:
# Fine-tuned Litus-ai (longseg)
local_model_path = "final_models/litus_longseg_ct2_1"

ft_results_litus_long = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)


display(ft_results_litus_long.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:08:23<00:00,  3.70it/s]



Global WER: 76.59% | Global CER: 22.65%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati potevano entrà ...,0.4167,0.0899
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",è estensivamente documentato e la documentazio...,0.5833,0.1264
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina l'ò fa e i nizion a d'accadira e mo...,0.6429,0.1429
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durant e' tal e' interval vento fura un'an ch'...,1.3571,0.3241
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,i du al analògh còntròllar e còmpatibili anche...,0.8571,0.1630


In [ ]:
# Fine-tuned Sandiago21 (shortseg)
local_model_path = "final_models/sandiago_shortseg_ct2_1"

ft_results_sandiago_short = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)

display(ft_results_sandiago_short.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:21:50<00:00,  3.09it/s]



Global WER: 27.11% | Global CER: 7.31%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,da questo luogo i predestinati potevano entrar...,0.0833,0.0112
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.1667,0.0230
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,la bambina lo fa e iniziano ad accadere molti ...,0.0714,0.0119
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche aggiunte ...,0.0000,0.0000
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il dual analog controller è compatibile anche ...,0.0000,0.0000


In [ ]:
# Fine-tuned Sandiago21 (longseg)
local_model_path = "final_models/sandiago_longseg_ct2_1"

ft_results_sandiago_long = transcribe_and_evaluate(
    model_or_path=local_model_path,
    df=test_mozilla_ita,
    audio_dir="cv-corpus-27.0-2026-09-11/it/clips",
    language="it",  
    beam_size=5,
)


display(ft_results_sandiago_long.head())

Transcribing & Evaluating: 100%|██████████| 15183/15183 [1:40:47<00:00,  2.51it/s]  



Global WER: 19.63% | Global CER: 5.28%


,audio_id,ground_truth,hypothesis,wer,cer
0,common_voice_it_26970003,Da questo luogo i predestinati potevano entrar...,Da questo luogo i predestinati potevano entrar...,0.0000,0.000
1,common_voice_it_26970001,"È estensivamente documentato, e la documentazi...",È estensivamente documentato e la documentazio...,0.1667,0.023
2,common_voice_it_26969995,La bambina lo fa e iniziano ad accadere molti ...,La bambina lo fa e iniziano ad accadere molti ...,0.0000,0.000
3,common_voice_it_26970006,Durante tale intervento furono anche aggiunte ...,Durante tale intervento furono anche aggiunte ...,0.0000,0.000
4,common_voice_it_26969998,Il Dual Analog Controller è compatibile anche ...,Il dual analog controller è compatibile anche ...,0.0000,0.000
